# 08 - Residual error rescue/harm analysis

Notebook này **không train lại model**. Nó nạp bốn prediction CSV đã lưu của Week 3 và so sánh trên cùng 400 ảnh validation, dùng ngưỡng cố định `0.5`.

Các comparison bắt buộc:
- Global vs Residual;
- Local vs Residual;
- Local + Residual (`LR_mean`) vs Local-only.

`rescue = baseline sai, candidate đúng`; `harm = baseline đúng, candidate sai`; `net gain = rescue - harm`. Kết quả ở đây là in-domain validation, không phải unseen-generator evaluation.

In [ ]:
from pathlib import Path
import sys

# Ưu tiên repo local/Kaggle; có thể ghi đè bằng các biến đường dẫn ở cell này.
local_candidates = [Path.cwd(), Path.cwd().parent, Path('/Users/nguyentran0703/Downloads/FaceTrace'), Path('/kaggle/working/FaceTrace')]
REPO_DIR = next((p for p in local_candidates if (p / 'src').is_dir()), Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

DEFAULT_ANALYSIS_ROOT = REPO_DIR / 'outputs/week03_kaggle_20261001/analysis/clean_seed42'
DEFAULT_VAL_MANIFEST = REPO_DIR / 'outputs/splits/val.csv'
DEFAULT_IMAGE_BASE = REPO_DIR
OUTPUT_DIR = DEFAULT_ANALYSIS_ROOT / 'error_rescue_harm'

# Nếu chạy trên Kaggle với artifact attach riêng, sửa bốn biến dưới đây.
ANALYSIS_ROOT = DEFAULT_ANALYSIS_ROOT
VAL_MANIFEST = DEFAULT_VAL_MANIFEST
IMAGE_PATH_BASE = DEFAULT_IMAGE_BASE
print('Repo:', REPO_DIR)
print('Analysis root:', ANALYSIS_ROOT)
print('Validation manifest:', VAL_MANIFEST)
print('Image base:', IMAGE_PATH_BASE)
print('Output:', OUTPUT_DIR)

In [ ]:
from src.evaluation.error_analysis import run_error_analysis

prediction_paths = {
    'global': ANALYSIS_ROOT / 'global_only.csv',
    'local': ANALYSIS_ROOT / 'local_only.csv',
    'residual': ANALYSIS_ROOT / 'residual.csv',
    'local_residual': ANALYSIS_ROOT / 'evaluation/LR_mean.csv',
}
missing = {name: str(path) for name, path in prediction_paths.items() if not path.is_file()}
if missing:
    raise FileNotFoundError(f'Missing prediction artifacts: {missing}')

result = run_error_analysis(
    prediction_paths=prediction_paths,
    validation_manifest=VAL_MANIFEST,
    image_path_base=IMAGE_PATH_BASE,
    output_dir=OUTPUT_DIR,
    threshold=0.5,
    seed=42,
)
protocol = __import__('json').loads((OUTPUT_DIR / 'run_manifest.json').read_text())
assert protocol['n_validation'] == 400, protocol['n_validation']
print(f"Validation images: {protocol['n_validation']}; comparison rows: {result['n_cases']} across {len(result['summaries'])} comparisons.")

In [ ]:
import pandas as pd

summary_rows = []
for item in result['summaries']:
    summary_rows.append({
        'comparison': item['comparison'],
        'n_images': item['n_images'],
        'baseline_AUROC': item['baseline_metrics']['auroc'],
        'candidate_AUROC': item['candidate_metrics']['auroc'],
        'baseline_BAcc': item['baseline_metrics']['balanced_accuracy'],
        'candidate_BAcc': item['candidate_metrics']['balanced_accuracy'],
        'rescue': item['rescue_count'],
        'harm': item['harm_count'],
        'net_gain': item['net_gain'],
    })
summary = pd.DataFrame(summary_rows)
groups = pd.read_csv(OUTPUT_DIR / 'error_rescue_harm.csv')
display(summary)
display(groups)

In [ ]:
from IPython.display import Image as DisplayImage, display

# Montage giữ nguyên ảnh RGB và ghi rõ group/probability; dùng để kiểm tra định tính.
for filename in ('residual_vs_global.png', 'residual_vs_local.png', 'local_residual_vs_local.png'):
    path = OUTPUT_DIR / 'montages' / filename
    if path.is_file():
        display(DisplayImage(filename=str(path)))
    else:
        print('Missing montage:', path)

## Artifact contract

Các file được tạo trong `error_rescue_harm/`:

- `error_rescue_harm.csv`: bốn group cho từng comparison;
- `error_cases.csv`: từng image ID, xác suất, predicted label và group;
- `metrics_by_comparison.csv`: AUROC/BAcc/F1/accuracy cho hai phía;
- `error_rescue_harm_summary.json`: summary và protocol;
- `run_manifest.json`: hash của các prediction CSV, threshold, seed và số ảnh;
- `montages/*.png`: các ảnh rescue/harm/both-correct/both-wrong được chọn deterministic.

Notebook chỉ xác nhận rescue/harm trên validation đã dùng để phát triển; không được diễn giải thành bằng chứng generalization sang generator chưa thấy.